# 03 · EGARCH(1,1)-t features (no look-ahead)

$$\log\sigma_t^2=\omega+\beta\log\sigma_{t-1}^2+\alpha\left(|z_{t-1}|-E|z|\right)+\gamma z_{t-1}$$

Fitted per stock to daily close-to-close returns (%), Student-t errors (Nelson, 1991).
$\gamma<0$ means bad news raises volatility more than good news (leverage effect).

**Fix compared with the earlier version:** EGARCH used to be estimated on the full 2016–2026 sample,
so the validation and test features were built with parameters that had already seen those years.
Here the parameters are estimated on **2016–2022 only**, then held fixed and run forward day by day.

Output features for day *t*: conditional volatility $\sigma_t$ and standardized shock $z_t$.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from volforecast import config as C, egarch

panel = pd.read_parquet(C.PROCESSED_DIR / "daily_panel.parquet")
universe = pd.read_csv(C.PROCESSED_DIR / "universe.csv")["Symbol"].tolist()
features, params = egarch.egarch_features(panel, universe)
features.to_parquet(C.PROCESSED_DIR / "egarch_features.parquet", index=False)
params.to_csv(C.TABLE_DIR / "egarch_parameters.csv", index=False)
print("Stocks:", len(params), "| converged:", params["converged"].sum())
print("Share with leverage effect (gamma < 0):", round((params["gamma[1]"] < 0).mean(), 2))
params.drop(columns=["Symbol", "converged"]).describe().T[["mean", "50%", "min", "max"]].round(3)

Stocks: 95 | converged: 95
Share with leverage effect (gamma < 0): 0.92


,mean,50%,min,max
mu,0.017,0.013,-0.221,0.223
omega,0.146,0.070,0.012,1.884
alpha[1],0.198,0.164,0.044,0.714
gamma[1],-0.042,-0.044,-0.142,0.053
beta[1],0.924,0.953,0.353,0.989
nu,4.783,4.703,2.656,8.482
n_fit,1646.611,1764.000,357.000,1764.000


In [2]:
features.describe().round(3)

,Date,EGARCH_Vol,EGARCH_Resid
count,232801,232801.000,232801.000
mean,2021-03-16 13:10:31.466360,2.115,0.020
min,2015-11-02 00:00:00,0.646,-12.727
25%,2018-08-28 00:00:00,1.611,-0.519
50%,2021-04-19 00:00:00,1.957,0.002
75%,2023-10-11 00:00:00,2.455,0.537
max,2026-04-09 00:00:00,28.122,16.255
std,NaN,0.754,0.971
